In [6]:
### Assignment #10 Gradient boosting

In [1]:
import numpy as np
import pandas as pd 
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelBinarizer
from xgboost import XGBClassifier

In [3]:
DATA_PATH = "https://raw.githubusercontent.com/Yorko/mlcourse.ai/main/data/"
train = pd.read_csv(DATA_PATH + "flight_delays_train.csv")
test = pd.read_csv(DATA_PATH + "flight_delays_test.csv")

train.head()

,Month,DayofMonth,DayOfWeek,DepTime,UniqueCarrier,Origin,Dest,Distance,dep_delayed_15min
0,c-8,c-21,c-7,1934,AA,ATL,DFW,732,N
1,c-4,c-20,c-3,1548,US,PIT,MCO,834,N
2,c-9,c-2,c-5,1422,XE,RDU,CLE,416,N
3,c-11,c-25,c-6,1015,OO,DEN,MEM,872,N
4,c-10,c-7,c-6,1828,WN,MDW,OMA,423,Y


In [5]:
train['Flight'] = train['Origin'] +'-->'+ train['Dest']
test['Flight'] = test['Origin'] +'-->'+test['Dest']

train[['Origin','Dest','Flight']].head()

,Origin,Dest,Flight
0,ATL,DFW,ATL-->DFW
1,PIT,MCO,PIT-->MCO
2,RDU,CLE,RDU-->CLE
3,DEN,MEM,DEN-->MEM
4,MDW,OMA,MDW-->OMA


In [8]:
categorical_cols = ['Month', 'DayofMonth', 'DayOfWeek', 'UniqueCarrier', 'Flight']

lb_dict = {}
train_ohe_parts = []
test_ohe_parts = []

for col in categorical_cols:
    lb = LabelBinarizer()
    train_ohe = lb.fit_transform(train[col])
    test_ohe = lb.transform(test[col])
    lb_dict[col] = lb
    train_ohe_parts.append(train_ohe)
    test_ohe_parts.append(test_ohe)
    print(col, "-> yeni sütun sayısı:", train_ohe.shape[1])

Month -> yeni sütun sayısı: 12
DayofMonth -> yeni sütun sayısı: 31
DayOfWeek -> yeni sütun sayısı: 7
UniqueCarrier -> yeni sütun sayısı: 22
Flight -> yeni sütun sayısı: 4429


In [9]:
from scipy.sparse import hstack, csr_matrix

# Sayısal feature'lar
train_num = train[['Distance', 'DepTime']].values
test_num = test[['Distance', 'DepTime']].values

# Tüm parçaları birleştir: sayısal + her kategorik OHE
X_train = hstack([csr_matrix(train_num)] + [csr_matrix(x) for x in train_ohe_parts]).tocsr()
X_test = hstack([csr_matrix(test_num)] + [csr_matrix(x) for x in test_ohe_parts]).tocsr()

y_train = train['dep_delayed_15min'].map({'Y': 1, 'N': 0}).values

print(X_train.shape)
print(X_test.shape)

(100000, 4503)
(100000, 4503)


In [11]:
X_train_part, X_valid, y_train_part, y_valid = train_test_split(
    X_train, y_train, test_size=0.3, random_state=17
)
train['dep_delayed_15min'].value_counts(normalize=True)

dep_delayed_15min
N    0.80956
Y    0.19044
Name: proportion, dtype: float64

In [12]:
logit = LogisticRegression(random_state=17, solver='liblinear')
logit.fit(X_train_part, y_train_part)
logit_valid_pred = logit.predict_proba(X_valid)[:, 1]

roc_auc_score(y_valid, logit_valid_pred)

0.6901826103025381

In [13]:
xgb_model = XGBClassifier(seed=17)
xgb_model.fit(X_train_part, y_train_part)
xgb_valid_pred = xgb_model.predict_proba(X_valid)[:, 1]

roc_auc_score(y_valid, xgb_valid_pred)

0.7217170232146863

In [14]:
w1 = 0.3  # logit ağırlığı, xgb ağırlığı 1-w1 = 0.7

blend_valid_pred = w1 * logit_valid_pred + (1 - w1) * xgb_valid_pred
roc_auc_score(y_valid, blend_valid_pred)

0.7206936206536048

In [15]:
for w1 in [0.0, 0.1, 0.2, 0.3, 0.4, 0.5]:
    blend_pred = w1 * logit_valid_pred + (1 - w1) * xgb_valid_pred
    score = roc_auc_score(y_valid, blend_pred)
    print(f"w1={w1}: {score:.4f}")

w1=0.0: 0.7217
w1=0.1: 0.7218
w1=0.2: 0.7214
w1=0.3: 0.7207
w1=0.4: 0.7194
w1=0.5: 0.7175


In [17]:
w1_best = 0.1

logit_full = LogisticRegression(random_state=17, solver='liblinear')
logit_full.fit(X_train, y_train)
logit_test_pred = logit_full.predict_proba(X_test)[:, 1]

xgb_full = XGBClassifier(seed=17)
xgb_full.fit(X_train, y_train)
xgb_test_pred = xgb_full.predict_proba(X_test)[:, 1]

final_test_pred = w1_best * logit_test_pred + (1 - w1_best) * xgb_test_pred

In [18]:
print(final_test_pred.shape)
print(final_test_pred[:5])

(100000,)
[0.04262271 0.04710425 0.05300193 0.19027894 0.190937  ]


In [19]:
submission = pd.DataFrame({
    'id': test.index,
    'dep_delayed_15min': final_test_pred
})

submission.to_csv('flight_delays_submission.csv', index=False)
submission.head()

,id,dep_delayed_15min
0,0,0.042623
1,1,0.047104
2,2,0.053002
3,3,0.190279
4,4,0.190937


In [21]:
!kaggle competitions submit -c flight-delays-spring-2018 -f flight_delays_submission.csv -m "XGBoost + LogReg blend with OHE features"

99 submissions remaining today.
Successfully submitted to Flight delays



  0%|          | 0.00/2.54M [00:00<?, ?B/s]
  1%|          | 16.0k/2.54M [00:00<01:00, 43.8kB/s]
  5%|4         | 128k/2.54M [00:00<00:07, 344kB/s]  
  8%|8         | 208k/2.54M [00:00<00:05, 475kB/s]
 12%|#2        | 320k/2.54M [00:00<00:03, 626kB/s]
 15%|#5        | 400k/2.54M [00:00<00:04, 515kB/s]
 18%|#7        | 464k/2.54M [00:01<00:04, 507kB/s]
 25%|##4       | 640k/2.54M [00:01<00:02, 799kB/s]
 31%|###       | 800k/2.54M [00:01<00:01, 993kB/s]
 36%|###5      | 928k/2.54M [00:01<00:01, 1.05MB/s]
 41%|####      | 1.03M/2.54M [00:01<00:01, 1.11MB/s]
 46%|####5     | 1.16M/2.54M [00:01<00:02, 689kB/s] 
 52%|#####1    | 1.31M/2.54M [00:01<00:01, 864kB/s]
 59%|#####9    | 1.50M/2.54M [00:02<00:01, 1.07MB/s]
 65%|######4   | 1.64M/2.54M [00:02<00:00, 1.11MB/s]
 70%|######9   | 1.77M/2.54M [00:02<00:00, 1.14MB/s]
 75%|#######4  | 1.89M/2.54M [00:02<00:00, 1.16MB/s]
 79%|#######9  | 2.02M/2.54M [00:02<00:00, 1.19MB/s]
 84%|########4 | 2.14M/2.54M [00:02<00:00, 1.20MB/s]
 89%|########9 

In [22]:
!kaggle competitions submissions -c flight-delays-spring-2018

     ref  fileName                      date                        description                               status                     publicScore  privateScore  
--------  ----------------------------  --------------------------  ----------------------------------------  -------------------------  -----------  ------------  
55884068  flight_delays_submission.csv  2026-08-30 06:50:07.980000  XGBoost + LogReg blend with OHE features  SubmissionStatus.COMPLETE  0.71755      0.71755       
